In [1]:
# Loading the raw dataset and performing initial inspection of data structure.

import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)

df = pd.read_csv("global_sales_performance.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7201 entries, 0 to 7200
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          7201 non-null   object 
 1   Date              7201 non-null   object 
 2   Region            7145 non-null   object 
 3   Country           7134 non-null   object 
 4   Product_Category  7153 non-null   object 
 5   Product_Name      7201 non-null   object 
 6   Units_Sold        7156 non-null   object 
 7   Unit_Price        7201 non-null   float64
 8   Total_Revenue     7201 non-null   object 
 9   Sales_Rep         7148 non-null   object 
 10  Currency          7151 non-null   object 
dtypes: float64(1), object(10)
memory usage: 619.0+ KB


In [2]:
#Taking a look at the data sample to make sure data loaded correctly.

import pandas as pd

df = pd.read_csv("global_sales_performance.csv")
df.head()

,Order_ID,Date,Region,Country,Product_Category,Product_Name,Units_Sold,Unit_Price,Total_Revenue,Sales_Rep,Currency
0,ORD-000001,2023-04-03,North America,United States,Food & Beverage,Organic Coffee Blend 1kg,58,20.10,1165.8,Valentina Lindqvist,USD
1,ORD-000002,2022-03-03,North America,Mexico,Automotive Parts,Ceramic Brake Pad Set,27,66.62,1798.74,Michael Zhang,MXN
2,ORD-000003,2024-05-09,North America,Mexico,Consumer Goods,Portable Blender,76,92.74,7048.24,David Smith,MXN
3,ORD-000004,2023-07-24,Europe,Netherlands,Automotive Parts,High-Flow Air Filter,48,113.37,5441.76,Ahmed Chen,EUR
4,ORD-000005,2023-06-26,Latin America,Brazil,Pharmaceuticals,Probiotic Complex,252,70.42,17745.84,Ingrid Al-Rashid,BRL


In [3]:
# Checking to make sure dimensions of data set are correct (7201 rows, 11 columns).

import pandas as pd

df = pd.read_csv("global_sales_performance.csv")
df.shape

(7201, 11)

In [4]:
# Counting duplicate rows.

print(df.duplicated().sum())

201


In [5]:
# Counting missing values in columns to see which needs attention.

print(df.isnull().sum())

Order_ID             0
Date                 0
Region              56
Country             67
Product_Category    48
Product_Name         0
Units_Sold          45
Unit_Price           0
Total_Revenue        0
Sales_Rep           53
Currency            50
dtype: int64


In [6]:
# Inspecting unique Region values to identify missing values, formatting inconsistencies.

print(df["Region"].unique())

['North America' 'Europe' 'Latin America' 'Middle East' nan 'Asia-Pacific']


In [7]:
#Checking for different or incorrect naming conventions for future standardization.

print(df["Country"].unique())

['United States' 'Mexico' 'Netherlands' 'Brazil' 'United Kingdom' 'Spain'
 'France' 'Peru' 'United Arab Emirates' 'Argentina' 'Turkey' 'Canada'
 'Colombia' 'Italy' 'Egypt' nan 'India' 'COLOMBIA' 'Japan' 'South Korea'
 'Singapore' 'Israel' 'Chile' 'Saudi Arabia' 'Germany' 'Australia' 'China'
 'NETHERLANDS' 'Emirates' 'CHILE' 'PRC' 'CANADA' 'U.S.' 'JAPAN'
 'ARGENTINA' 'ITALY' 'EGYPT' 'SOUTH KOREA' 'GER' 'SINGAPORE' 'PERU'
 'SPAIN' 'MEXICO' 'United States of America' 'Germany ' 'TURKEY'
 'Australia ' 'INDIA' 'U.A.E.' 'ISRAEL' 'UAE' 'S. Arabia' 'England' 'USA'
 'Deutschland' 'UK' 'CHINA' 'Brasil' 'BRA' 'Saudi Arabia ' 'FRANCE' 'KSA']


In [8]:
# Created a statistical summary to identify any outliers.

print(df.describe())

        Unit_Price
count     7,201.00
mean      1,499.09
std      23,701.08
min           8.59
25%          67.94
50%         184.14
75%         487.66
max   1,151,535.06


In [9]:
# Checking the shape before change.
print("Before:", df.shape)

# Removing duplicate rows.
df = df.drop_duplicates()

# Checking the shape after change.
print("After:", df.shape)

Before: (7201, 11)
After: (7000, 11)


In [10]:
# Sales_Rep column is not critical to the analysis, rows can be retained and the missing
# values set to "Unknown".

df["Sales_Rep"] = df["Sales_Rep"].fillna("Unknown")

In [11]:
# Built a country-to-region reference dictionary to fill missing Region values.
# Uses a lambda function with axis=1 to scan across columns in the same row.
# Only fills Region if it is currently missing, leaving existing values unchanged.

country_region = {
    "United States": "North America", "Canada": "North America", "Mexico": "North America",
    "Germany": "Europe", "France": "Europe", "United Kingdom": "Europe",
    "Italy": "Europe", "Spain": "Europe", "Netherlands": "Europe",
    "Japan": "Asia-Pacific", "China": "Asia-Pacific", "Australia": "Asia-Pacific",
    "South Korea": "Asia-Pacific", "India": "Asia-Pacific", "Singapore": "Asia-Pacific",
    "Brazil": "Latin America", "Argentina": "Latin America", "Colombia": "Latin America",
    "Chile": "Latin America", "Peru": "Latin America",
    "United Arab Emirates": "Middle East", "Saudi Arabia": "Middle East",
    "Israel": "Middle East", "Turkey": "Middle East", "Egypt": "Middle East"
}

df["Region"] = df.apply(
    lambda row: country_region.get(row["Country"], row["Region"])
    if pd.isnull(row["Region"]) else row["Region"], axis=1
)

In [12]:
# Built a country-to-currency reference dictionary similar to the previous one
# including the same approach of using lambda to fill missing values.

country_currency = {
    "United States": "USD", "Canada": "CAD", "Mexico": "MXN",
    "Germany": "EUR", "France": "EUR", "United Kingdom": "GBP",
    "Italy": "EUR", "Spain": "EUR", "Netherlands": "EUR",
    "Japan": "JPY", "China": "CNY", "Australia": "AUD",
    "South Korea": "KRW", "India": "INR", "Singapore": "SGD",
    "Brazil": "BRL", "Argentina": "ARS", "Colombia": "COP",
    "Chile": "CLP", "Peru": "PEN",
    "United Arab Emirates": "AED", "Saudi Arabia": "SAR",
    "Israel": "ILS", "Turkey": "TRY", "Egypt": "EGP"
}

df["Currency"] = df.apply(
    lambda row: country_currency.get(row["Country"], row["Currency"])
    if pd.isnull(row["Currency"]) else row["Currency"], axis=1
)

In [13]:
# Dropping rows where Country, Product_Category and Units_Sold could not be determined and 
# would be harmful to analysis if left in.

df = df.dropna(subset=["Country", "Product_Category", "Units_Sold"])

In [14]:
# Checking to ensure all missing values are dropped.

print(df.isnull().sum())
print("Shape after missing value handling:", df.shape)

Order_ID            0
Date                0
Region              0
Country             0
Product_Category    0
Product_Name        0
Units_Sold          0
Unit_Price          0
Total_Revenue       0
Sales_Rep           0
Currency            0
dtype: int64
Shape after missing value handling: (6840, 11)


In [15]:
# Verifying that the country-to-region dictionary worked and discovered remaining variations.
# Remaining variations to be handled next.

print(sorted(df["Country"].unique()))

['ARGENTINA', 'Argentina', 'Australia', 'Australia ', 'BRA', 'Brasil', 'Brazil', 'CANADA', 'CHILE', 'CHINA', 'COLOMBIA', 'Canada', 'Chile', 'China', 'Colombia', 'Deutschland', 'EGYPT', 'Egypt', 'Emirates', 'England', 'FRANCE', 'France', 'GER', 'Germany', 'Germany ', 'INDIA', 'ISRAEL', 'ITALY', 'India', 'Israel', 'Italy', 'JAPAN', 'Japan', 'KSA', 'MEXICO', 'Mexico', 'NETHERLANDS', 'Netherlands', 'PERU', 'PRC', 'Peru', 'S. Arabia', 'SINGAPORE', 'SOUTH KOREA', 'SPAIN', 'Saudi Arabia', 'Saudi Arabia ', 'Singapore', 'South Korea', 'Spain', 'TURKEY', 'Turkey', 'U.A.E.', 'U.S.', 'UAE', 'UK', 'USA', 'United Arab Emirates', 'United Kingdom', 'United States', 'United States of America']


In [16]:
# Standardizing country naming variations for easier readability and to prevent separate
# counts for the same country. 

country_standardization = {
    # United States variants
    "USA": "United States", "U.S.": "United States",
    "US": "United States", "United States of America": "United States",
    # United Kingdom variants
    "UK": "United Kingdom", "U.K.": "United Kingdom",
    "Great Britain": "United Kingdom", "England": "United Kingdom",
    # United Arab Emirates variants
    "UAE": "United Arab Emirates", "U.A.E.": "United Arab Emirates",
    "Emirates": "United Arab Emirates",
    # All other country variants
    "Deutschland": "Germany", "GER": "Germany", "Germany ": "Germany",
    "PRC": "China", "China ": "China", "CHINA": "China",
    "AUS": "Australia", "Australia ": "Australia",
    "Brasil": "Brazil", "BRA": "Brazil", "Brazil ": "Brazil",
    "KSA": "Saudi Arabia", "Saudi Arabia ": "Saudi Arabia", "S. Arabia": "Saudi Arabia"
}

df["Country"] = df["Country"].replace(country_standardization)

In [17]:
text_columns = ["Region", "Country", "Product_Category", "Product_Name", "Sales_Rep", "Currency"]

# Initial whitespace removal across columns that may have it.

for col in text_columns:
    df[col] = df[col].str.strip()

In [18]:
# Checking if standardization worked, which leaves only upper case variations to be corrected
# next.

print(sorted(df["Country"].unique()))

['ARGENTINA', 'Argentina', 'Australia', 'Brazil', 'CANADA', 'CHILE', 'COLOMBIA', 'Canada', 'Chile', 'China', 'Colombia', 'EGYPT', 'Egypt', 'FRANCE', 'France', 'Germany', 'INDIA', 'ISRAEL', 'ITALY', 'India', 'Israel', 'Italy', 'JAPAN', 'Japan', 'MEXICO', 'Mexico', 'NETHERLANDS', 'Netherlands', 'PERU', 'Peru', 'SINGAPORE', 'SOUTH KOREA', 'SPAIN', 'Saudi Arabia', 'Singapore', 'South Korea', 'Spain', 'TURKEY', 'Turkey', 'United Arab Emirates', 'United Kingdom', 'United States']


In [19]:
# Cleaning up the country names, first with casing.
df["Country"] = df["Country"].str.title()

# Now applying dictionary standardization
df["Country"] = df["Country"].replace(country_standardization)

# Running a separate whitespace removal to catch anything that may have been missed from Cell 17.
df["Country"] = df["Country"].str.strip()

# Verification check.
print(sorted(df["Country"].unique()))

['Argentina', 'Australia', 'Brazil', 'Canada', 'Chile', 'China', 'Colombia', 'Egypt', 'France', 'Germany', 'India', 'Israel', 'Italy', 'Japan', 'Mexico', 'Netherlands', 'Peru', 'Saudi Arabia', 'Singapore', 'South Korea', 'Spain', 'Turkey', 'United Arab Emirates', 'United Kingdom', 'United States']


In [20]:
# Checking data types to make sure they are correctly assigned. Multiple columns
# incorrectly listed and require conversion.

print(df.dtypes)

Order_ID             object
Date                 object
Region               object
Country              object
Product_Category     object
Product_Name         object
Units_Sold           object
Unit_Price          float64
Total_Revenue        object
Sales_Rep            object
Currency             object
dtype: object


In [21]:
# Changed Date to date/time format. The errors="coerce" code converts any unparsable
# values to NaT, rather than raising an 'error'.

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [22]:
# Change Units_Sold to numeric format, complimented with errors="coerce".

df["Units_Sold"] = df["Units_Sold"].astype(str).str.replace("units", "", case=False).str.strip()
df["Units_Sold"] = pd.to_numeric(df["Units_Sold"], errors="coerce")

In [23]:
# Change Total_Revenue to numeric format,complimented with errors="coerce".

df["Total_Revenue"] = df["Total_Revenue"].astype(str).str.replace("$", "", regex=False).str.strip()
df["Total_Revenue"] = pd.to_numeric(df["Total_Revenue"], errors="coerce")

In [24]:
# Confirming all three columns now have correct data types after conversion.

print(df.dtypes)

Order_ID                    object
Date                datetime64[ns]
Region                      object
Country                     object
Product_Category            object
Product_Name                object
Units_Sold                   int64
Unit_Price                 float64
Total_Revenue              float64
Sales_Rep                   object
Currency                    object
dtype: object


In [25]:
# Checking to see if any missing or null values have been introduced with the 
# data type conversion.

print(df[["Date", "Units_Sold", "Total_Revenue"]].isnull().sum())

Date             30
Units_Sold        0
Total_Revenue     0
dtype: int64


In [26]:
# All are displaying NaT and cannot be recovered, dropping these rows next.

print(df[df["Date"].isnull()]["Date"].head(20))

10     NaT
409    NaT
414    NaT
490    NaT
710    NaT
951    NaT
955    NaT
1130   NaT
1149   NaT
1538   NaT
1726   NaT
1854   NaT
2447   NaT
2736   NaT
2757   NaT
2832   NaT
2895   NaT
3661   NaT
4396   NaT
4709   NaT
Name: Date, dtype: datetime64[ns]


In [27]:
# 30 rows now have missing date data. Date is essential for the analysis and
# cannot be reliably reconstructed.

print("Current shape:", df.shape)
print("Rows with missing Date:", df["Date"].isnull().sum())

Current shape: (6840, 11)
Rows with missing Date: 30


In [28]:
# Dropping rows that now contain invalid dates.

df = df.dropna(subset=["Date"])
print("Shape after dropping invalid dates:", df.shape)

Shape after dropping invalid dates: (6810, 11)


In [29]:
# One final verification check to ensure data correction.

print(df.dtypes)

Order_ID                    object
Date                datetime64[ns]
Region                      object
Country                     object
Product_Category            object
Product_Name                object
Units_Sold                   int64
Unit_Price                 float64
Total_Revenue              float64
Sales_Rep                   object
Currency                    object
dtype: object


In [30]:
# There might be negative revenues values and 0 values that have appeared, and
# perhaps some extreme price outliers now after the data type correction.

print("Negative Revenue rows:", (df["Total_Revenue"] < 0).sum())

print("Zero Units_Sold rows:", (df["Units_Sold"] == 0).sum())

# Checking to see if the extreme outlier distribution is significant.
print(df["Unit_Price"].describe())

Negative Revenue rows: 30
Zero Units_Sold rows: 71
count       6,810.00
mean        1,535.57
std        24,366.74
min             8.59
25%            68.61
50%           185.59
75%           489.54
max     1,151,535.06
Name: Unit_Price, dtype: float64


In [31]:
# Inspecting top 20 highest prices to understand full scale of outlier values.

print(df["Unit_Price"].sort_values(ascending=False).head(20))

6650   1,151,535.06
2751   1,144,905.48
6223     995,555.82
420      586,088.47
6507     191,348.07
3005      52,103.37
6380      51,346.37
3382      45,121.88
5513      44,584.75
2492      39,732.46
6390      33,140.48
2001      30,965.18
5324      30,827.52
5038      30,379.79
3388      28,727.72
7121      25,643.12
1442      23,715.11
534       22,912.22
7119      22,094.97
3508      18,113.23
Name: Unit_Price, dtype: float64


In [32]:
# Dropping rows that contain a total revenue of 0. This cannot be reliably reconstructed
# for the analysis and is invalid.

df = df[df["Total_Revenue"] >= 0]
print("Shape after removing negative revenue:", df.shape)

Shape after removing negative revenue: (6780, 11)


In [33]:
# Dropping rows that contain 0 units sold. A transaction with no units sold is invalid.

df = df[df["Units_Sold"] > 0]
print("Shape after removing zero units:", df.shape)

Shape after removing zero units: (6709, 11)


In [34]:
# With this being a synthetic data set, the price range variation is making large skews
# that could be interpreted as extreme compared to ordinary business datasets. Due to the
# variation in product types and their prices, an IQR with double the standard multiplier 
# should be suffienct to retain correct price data and only catch extreme outlier values.

Q1 = df["Unit_Price"].quantile(0.25)
Q3 = df["Unit_Price"].quantile(0.75)
IQR = Q3 - Q1

upper_bound = Q3 + (3.0 * IQR)

print(f"Q1: {Q1}")
print(f"Q3: {Q3}")
print(f"IQR: {IQR}")
print(f"Upper bound: {upper_bound}")
print(f"Rows flagged as extreme outliers: {(df['Unit_Price'] > upper_bound).sum()}")

Q1: 68.36
Q3: 485.61
IQR: 417.25
Upper bound: 1737.3600000000001
Rows flagged as extreme outliers: 703


In [35]:
# Checking the minimum, maximum, and average pricies for products across all categories.

category_price_stats = df.groupby("Product_Category")["Unit_Price"].agg(["min", "mean", "max"]).round(2).sort_values("max", ascending=False)

print(category_price_stats)

                        min      mean          max
Product_Category                                  
Industrial Equipment 458.28 10,738.99 1,151,535.06
Raw Materials         51.31    533.65    51,346.37
Automotive Parts      28.82    419.51    45,121.88
Consumer Goods        17.12    306.70    39,732.46
Electronics           76.31    563.40    30,965.18
Apparel & Footwear    20.37    309.52    30,379.79
Food & Beverage        8.59     74.41    10,258.73
Pharmaceuticals       10.69     54.41     2,933.59


In [36]:
# 703 rows is still too much and might be linked to the categories, and decided to make
# a cap on category prices. Normally this would be done with either consultation 
# from an expert in the subject matter or go through previously recorded historical business data, 
# but for this project I came up with a reasonable cap based on the mean and maximum price listed in the dataset.

category_price_caps = {
    "Electronics": 2000,
    "Apparel & Footwear": 500,
    "Industrial Equipment": 25000,
    "Consumer Goods": 800,
    "Food & Beverage": 200,
    "Pharmaceuticals": 200,
    "Automotive Parts": 1200,
    "Raw Materials": 2000
}

# Check to see how many are flagged after implementing the cap.
outlier_mask = df.apply(
    lambda row: row["Unit_Price"] > category_price_caps.get(row["Product_Category"], float("inf")),
    axis=1
)

print("Rows exceeding category price caps:", outlier_mask.sum())

Rows exceeding category price caps: 37


In [37]:
# Dropping the outlier rows. "~" inverts the mask to retain rows where the condition is
# False.

df = df[~outlier_mask]
print("Shape after removing extreme Unit_Price outliers:", df.shape)

Shape after removing extreme Unit_Price outliers: (6672, 11)


In [38]:
# Final verification to confirm there is no negative revenue, zero units
# or price outliers.

print("Negative Revenue remaining:", (df["Total_Revenue"] < 0).sum())
print("Zero Units_Sold remaining:", (df["Units_Sold"] == 0).sum())
print("Unit_Price outliers remaining:", df.apply(
    lambda row: row["Unit_Price"] > category_price_caps.get(row["Product_Category"], float("inf")),
    axis=1).sum())

Negative Revenue remaining: 0
Zero Units_Sold remaining: 0
Unit_Price outliers remaining: 0


In [39]:
# Recalculating the Total_Revenue from Units_Sold multipled by Unit_Price to ensure
# consistency across all rows after the cleaning.

df["Total_Revenue"] = df["Units_Sold"] * df["Unit_Price"]
df["Total_Revenue"] = df["Total_Revenue"].round(2)

In [40]:
# Resetting the dataframe index after row removals to ensure clean sequencing.

df = df.reset_index(drop=True)

In [41]:
# Final check to confirm the shape, data types, missing values, and sample rows are handled.

print("Final shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nSample rows:")
df.head()

Final shape: (6672, 11)

Data types:
 Order_ID                    object
Date                datetime64[ns]
Region                      object
Country                     object
Product_Category            object
Product_Name                object
Units_Sold                   int64
Unit_Price                 float64
Total_Revenue              float64
Sales_Rep                   object
Currency                    object
dtype: object

Missing values:
 Order_ID            0
Date                0
Region              0
Country             0
Product_Category    0
Product_Name        0
Units_Sold          0
Unit_Price          0
Total_Revenue       0
Sales_Rep           0
Currency            0
dtype: int64

Sample rows:


,Order_ID,Date,Region,Country,Product_Category,Product_Name,Units_Sold,Unit_Price,Total_Revenue,Sales_Rep,Currency
0,ORD-000001,2023-04-03,North America,United States,Food & Beverage,Organic Coffee Blend 1kg,58,20.10,"1,165.80",Valentina Lindqvist,USD
1,ORD-000002,2022-03-03,North America,Mexico,Automotive Parts,Ceramic Brake Pad Set,27,66.62,"1,798.74",Michael Zhang,MXN
2,ORD-000003,2024-05-09,North America,Mexico,Consumer Goods,Portable Blender,76,92.74,"7,048.24",David Smith,MXN
3,ORD-000004,2023-07-24,Europe,Netherlands,Automotive Parts,High-Flow Air Filter,48,113.37,"5,441.76",Ahmed Chen,EUR
4,ORD-000005,2023-06-26,Latin America,Brazil,Pharmaceuticals,Probiotic Complex,252,70.42,"17,745.84",Ingrid Al-Rashid,BRL


In [42]:
# Saving notebook.

print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print(df.columns.tolist())
df.to_csv("Global_Sales_Performance_Cleaned.csv", index=False)
print("Saved successfully.")

Final shape: (6672, 11)
Missing values: 0
['Order_ID', 'Date', 'Region', 'Country', 'Product_Category', 'Product_Name', 'Units_Sold', 'Unit_Price', 'Total_Revenue', 'Sales_Rep', 'Currency']
Saved successfully.
